In [1]:
"""
Parent Document Retriever (PDR) -- Production-Grade RAG Pipeline
=================================================================
Architecture   : Parent Document Retriever (LangChain)
LLM            : Azure OpenAI (GPT-4o or GPT-4-turbo via AzureChatOpenAI)
Embeddings     : HuggingFace sentence-transformers/all-MiniLM-L6-v2  (local, no API key)
Vector Store   : ChromaDB  (persistent, child chunks indexed here)
Doc Store      : InMemoryStore  (parent documents stored here)
Data Source    : Three publicly available research PDFs from arXiv

Reference PDFs (all open-access, freely downloadable):
    1. "Attention Is All You Need" -- Vaswani et al., 2017
       https://arxiv.org/pdf/1706.03762.pdf

    2. "BERT: Pre-training of Deep Bidirectional Transformers" -- Devlin et al., 2018
       https://arxiv.org/pdf/1810.04805.pdf

    3. "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks" -- Lewis et al., 2020
       https://arxiv.org/pdf/2005.11401.pdf

Core Concept -- WHY Parent Document Retriever:
    Standard RAG chunks documents into small pieces for precise embedding-level
    similarity. But small chunks lose surrounding context, so the LLM sees a
    sentence fragment instead of a complete passage, leading to incomplete answers.

    PDR solves this by maintaining TWO levels of granularity:
        - Child chunks  (small, e.g., 200 chars)  --> embedded and stored in ChromaDB
          for high-precision semantic retrieval.
        - Parent chunks (large, e.g., 1500 chars) --> stored in a docstore;
          retrieved in full when a matching child is found.

    At query time: embed query --> find top-k child chunks --> fetch their parent
    documents from the docstore --> send full parent text to the LLM.

    Result: retrieval precision of small chunks PLUS the rich context of large
    parent documents in the generation step.

Pipeline Flow:
    PDF Download (arXiv) --> PyPDFLoader --> Parent split (1500 chars)
    --> Child split (200 chars, tagged with parent_id)
    --> HuggingFace embed child chunks --> ChromaDB
    --> InMemoryStore holds parent docs

    Query --> embed --> ChromaDB child search --> parent_id lookup
    --> fetch parents from InMemoryStore --> prompt + Azure OpenAI --> answer

"""

'\nParent Document Retriever (PDR) -- Production-Grade RAG Pipeline\n=================================================================\nArchitecture   : Parent Document Retriever (LangChain)\nLLM            : Azure OpenAI (GPT-4o or GPT-4-turbo via AzureChatOpenAI)\nEmbeddings     : HuggingFace sentence-transformers/all-MiniLM-L6-v2  (local, no API key)\nVector Store   : ChromaDB  (persistent, child chunks indexed here)\nDoc Store      : InMemoryStore  (parent documents stored here)\nData Source    : Three publicly available research PDFs from arXiv\n\nReference PDFs (all open-access, freely downloadable):\n    1. "Attention Is All You Need" -- Vaswani et al., 2017\n       https://arxiv.org/pdf/1706.03762.pdf\n\n    2. "BERT: Pre-training of Deep Bidirectional Transformers" -- Devlin et al., 2018\n       https://arxiv.org/pdf/1810.04805.pdf\n\n    3. "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks" -- Lewis et al., 2020\n       https://arxiv.org/pdf/2005.11401.pdf\n\n

In [2]:
# !pip install langchain chromadb sentence-transformers PyPDF2 

In [40]:
!pip show langchain

Name: langchain
Version: 1.2.13
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: C:\Users\dhanu\miniconda3\Lib\site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 


In [4]:
# !pip install --upgrade --force-reinstall langchain langchain-core langchain-community langchain-chroma langchain-huggingface langchain-openai

  Using cached langchain-1.2.13-py3-none-any.whl.metadata (5.8 kB)
  Using cached langchain_core-1.2.22-py3-none-any.whl.metadata (4.4 kB)
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_chroma-1.1.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached langchain_huggingface-1.2.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached langchain_openai-1.1.12-py3-none-any.whl.metadata (3.1 kB)
  Using cached langgraph-1.1.3-py3-none-any.whl.metadata (7.4 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached langsmith-0.7.22-py3-none-any.whl.metadata (15 kB)
  Using cached packaging-26.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-win_amd64.whl.metadata (2.4 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached uu

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
agentops 0.4.21 requires packaging<25.0,>=21.0, but you have packaging 26.0 which is incompatible.
autogen-core 0.7.5 requires protobuf~=5.29.3, but you have protobuf 6.33.6 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
grpcio-health-checking 1.66.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
grpcio-tools 1.66.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
jsonschema-path 0.3.4 requires referencing<0.37.0, but you have referencing 0.37.0 which is incompatible.
opencv-pyth

In [5]:
# !pip install langchain-core

In [6]:
# !pip install langchain-community --upgrade --force-reinstall

In [1]:
from langchain_core.stores import InMemoryStore

# Initialize the store
store = InMemoryStore()

In [8]:
# !pip install --upgrade langchain langchain-community langchain-core

In [3]:
from langchain_classic.retrievers import (
    ParentDocumentRetriever,
    ContextualCompressionRetriever,
    MergerRetriever,
)
from langchain_classic.retrievers.document_compressors import DocumentCompressorPipeline

C:\Users\dhanu\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0326 22:07:17.987000 37148 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [25]:
import os 
import logging 
import sys 
import hashlib 
import time
import urllib.request 
from typing import List,Optional
from pathlib import Path

# Core schema
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.stores import InMemoryStore

# Retrievers
from langchain_classic.retrievers import (
    ParentDocumentRetriever,
    ContextualCompressionRetriever,
    MergerRetriever,
)

# Vector stores & embeddings
from langchain_chroma import Chroma

# Document loaders
from langchain_community.document_loaders import PyPDFLoader

# Models
from langchain_openai import AzureChatOpenAI
from langchain_community.embeddings import HuggingFaceEmbeddings

# Prompting & parsing
from langchain_core.prompts import ChatPromptTemplate

from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [8]:
# ===========================================================================
# LOGGING  -- structured, level-controlled, timestamped
# ===========================================================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("parent_doc_retriever_rag")


In [9]:


# ===========================================================================
# SECTION 1 -- CONFIGURATION
# All tunable hyper-parameters are centralized in one class.
# ===========================================================================

class PDRConfig:
    """
    Centralized configuration for the Parent Document Retriever RAG pipeline.

    Splitting strategy:
        PARENT_CHUNK_SIZE  : size of the document unit returned to the LLM.
                             Large enough to include complete explanations and
                             context around the matched child chunk.
        CHILD_CHUNK_SIZE   : size of the document unit embedded in ChromaDB.
                             Small enough for precise semantic similarity matching.
                             Rule of thumb: child is ~1/6 to 1/8 of parent.
        CHILD_CHUNK_OVERLAP: overlap across adjacent child chunks to prevent
                             boundary context loss during retrieval.
    """

    # ---- Dataset: publicly available research PDFs (arXiv open-access) ----
    # Each tuple is (descriptive_name, direct_pdf_url)
    PDF_SOURCES: List[tuple] = [
        (
            "attention_is_all_you_need",
            "https://arxiv.org/pdf/1706.03762.pdf",
        ),
        (
            "bert_pretraining_transformers",
            "https://arxiv.org/pdf/1810.04805.pdf",
        ),
        (
            "rag_knowledge_intensive_nlp",
            "https://arxiv.org/pdf/2005.11401.pdf",
        ),
    ]
    PDF_DOWNLOAD_DIR: str = "./pdfs"            # local folder for downloaded PDFs

    # ---- Text splitting strategy ----------------------------------------
    PARENT_CHUNK_SIZE: int = 1500               # characters; sent to LLM as context
    PARENT_CHUNK_OVERLAP: int = 150             # overlap preserves cross-boundary context
    CHILD_CHUNK_SIZE: int = 200                 # characters; embedded in ChromaDB
    CHILD_CHUNK_OVERLAP: int = 30               # small overlap sufficient for short chunks

    # ---- Embeddings -------------------------------------------------------
    EMBEDDING_MODEL: str = "sentence-transformers/all-MiniLM-L6-v2"
    EMBEDDING_DEVICE: str = "cpu"               # "cuda" for GPU acceleration

    # ---- ChromaDB ---------------------------------------------------------
    CHROMA_PERSIST_DIR: str = "./chroma_pdr_db"
    CHROMA_COLLECTION: str = "pdr_child_chunks"

    # ---- Azure OpenAI LLM -------------------------------------------------
    # All values read from environment variables for security (never hardcode keys)
    AZURE_OPENAI_API_KEY: str       = os.environ.get("AZURE_OPENAI_API_KEY", "")
    AZURE_OPENAI_ENDPOINT: str      = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
    AZURE_OPENAI_DEPLOYMENT: str    = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-4o")
    AZURE_OPENAI_API_VERSION: str   = os.environ.get("AZURE_OPENAI_API_VERSION", "2024-02-15-preview")
    LLM_TEMPERATURE: float          = 0.0       # deterministic for factual Q&A
    LLM_MAX_TOKENS: int             = 1024

    # ---- Retrieval --------------------------------------------------------
    RETRIEVER_K: int = 3    # number of parent documents returned per query

    # ---- Prompt template --------------------------------------------------
    RAG_PROMPT_TEMPLATE: str = """You are a precise technical research assistant.
Use ONLY the information provided in the context below to answer the question.
If the answer cannot be found in the context, respond with:
"The provided documents do not contain enough information to answer this question."

Do NOT use prior knowledge beyond what is explicitly stated in the context.

Context (retrieved from research papers):
{context}

Question: {question}

Provide a clear, technically accurate, and well-structured answer:"""



In [10]:

# ===========================================================================
# SECTION 2 -- PDF DOWNLOADER
# Downloads PDFs from public arXiv URLs into a local directory.
# ===========================================================================

def download_pdfs(config: PDRConfig) -> List[Path]:
    """
    Download research PDFs from arXiv to the local filesystem.

    arXiv is fully open-access. PDFs downloaded from https://arxiv.org/pdf/<id>.pdf
    are publicly available under the authors' chosen license. This function
    implements polite downloading: it skips files that already exist (cached),
    adds a 1-second delay between requests to avoid rate-limiting, and verifies
    the download completed by checking file size.

    Args:
        config: PDRConfig with PDF_SOURCES list and PDF_DOWNLOAD_DIR path.

    Returns:
        List of Path objects pointing to successfully downloaded PDF files.

    Raises:
        RuntimeError: If a download fails after the attempt.
    """
    download_dir = Path(config.PDF_DOWNLOAD_DIR)
    download_dir.mkdir(parents=True, exist_ok=True)

    downloaded_paths: List[Path] = []

    for name, url in config.PDF_SOURCES:
        local_path = download_dir / f"{name}.pdf"

        if local_path.exists() and local_path.stat().st_size > 10_000:
            # File already cached and non-trivial in size -- skip download
            logger.info("Cached PDF found: %s (%.1f KB)", local_path.name, local_path.stat().st_size / 1024)
            downloaded_paths.append(local_path)
            continue

        logger.info("Downloading: %s --> %s", url, local_path)
        try:
            # Set a descriptive User-Agent as per arXiv's usage policy
            request = urllib.request.Request(
                url,
                headers={
                    "User-Agent": (
                        "Mozilla/5.0 (compatible; RAG-Research-Pipeline/1.0; "
                        "+https://github.com/research/rag)"
                    )
                },
            )
            with urllib.request.urlopen(request, timeout=60) as response:
                pdf_bytes = response.read()

            if len(pdf_bytes) < 10_000:
                raise RuntimeError(
                    f"Downloaded file is suspiciously small ({len(pdf_bytes)} bytes). "
                    f"URL may have redirected or returned an error: {url}"
                )

            local_path.write_bytes(pdf_bytes)
            logger.info(
                "Downloaded: %s (%.1f KB)", local_path.name, len(pdf_bytes) / 1024
            )
            downloaded_paths.append(local_path)

            # Polite delay between downloads
            time.sleep(1.0)

        except Exception as exc:
            raise RuntimeError(
                f"Failed to download PDF from '{url}'. "
                "Check your internet connection or try manually downloading "
                f"and placing the file at '{local_path}'."
            ) from exc

    return downloaded_paths


In [11]:

# ===========================================================================
# SECTION 3 -- PDF LOADING
# Load each PDF file and tag documents with source metadata.
# ===========================================================================

def load_pdf_documents(pdf_paths: List[Path]) -> List[Document]:
    """
    Load PDF files using LangChain's PyPDFLoader and attach source metadata.

    PyPDFLoader uses PyPDF2 under the hood. It loads each page as a separate
    Document. We merge all pages belonging to the same PDF into a single list,
    tagging each Document with:
        - source     : the PDF filename
        - paper_name : human-readable paper title derived from the filename
        - page       : page number within the original document

    This metadata flows through the splitting and indexing pipeline and is
    preserved in ChromaDB, enabling source attribution in answers.

    Args:
        pdf_paths: List of Path objects pointing to downloaded PDF files.

    Returns:
        Flat list of LangChain Documents, one per page across all PDFs.
    """
    all_docs: List[Document] = []

    for pdf_path in pdf_paths:
        logger.info("Loading PDF: %s", pdf_path.name)

        try:
            loader = PyPDFLoader(str(pdf_path))
            pages = loader.load()
        except Exception as exc:
            logger.error("Failed to load PDF '%s': %s", pdf_path.name, exc)
            raise

        # Enrich metadata with paper name derived from filename
        paper_name = pdf_path.stem.replace("_", " ").title()
        for doc in pages:
            doc.metadata["source"] = pdf_path.name
            doc.metadata["paper_name"] = paper_name
            # PyPDFLoader already adds 'page' key (0-indexed); keep it

        logger.info("  Loaded %d pages from '%s'", len(pages), pdf_path.name)
        all_docs.extend(pages)

    logger.info("Total pages loaded across all PDFs: %d", len(all_docs))
    return all_docs


In [12]:


# ===========================================================================
# SECTION 4 -- EMBEDDING MODEL INITIALIZATION
# ===========================================================================

def build_embeddings(config: PDRConfig) -> HuggingFaceEmbeddings:
    """
    Initialize the HuggingFace sentence-transformers embedding model.

    all-MiniLM-L6-v2 characteristics:
        - Architecture : 6-layer MiniLM distilled from a large BERT model
        - Output dims  : 384-dimensional unit-normalized vectors
        - Speed        : ~14K sentences/sec on CPU; very fast inference
        - Quality      : Competitive with larger models on MTEB semantic similarity
        - License      : Apache 2.0 (free for commercial use)

    normalize_embeddings=True ensures unit-norm vectors, which is required
    for cosine similarity to be mathematically correct in ChromaDB.

    Args:
        config: PDRConfig with EMBEDDING_MODEL and EMBEDDING_DEVICE settings.

    Returns:
        Initialized HuggingFaceEmbeddings instance.
    """
    logger.info(
        "Initializing embedding model: %s (device=%s)",
        config.EMBEDDING_MODEL, config.EMBEDDING_DEVICE,
    )
    embeddings = HuggingFaceEmbeddings(
        model_name=config.EMBEDDING_MODEL,
        model_kwargs={"device": config.EMBEDDING_DEVICE},
        encode_kwargs={"normalize_embeddings": True},
    )
    logger.info("Embedding model ready.")
    return embeddings


In [26]:


# ===========================================================================
# SECTION 5 -- PARENT DOCUMENT RETRIEVER CONSTRUCTION
# The heart of PDR: dual-store architecture with separate splitters for
# parent and child chunks.
# ===========================================================================

def build_parent_document_retriever(
    documents: List[Document],
    embeddings: HuggingFaceEmbeddings,
    config: PDRConfig,
) -> ParentDocumentRetriever:
    """
    Build and populate the Parent Document Retriever.

    Internal architecture of ParentDocumentRetriever:
        - docstore      : InMemoryStore  -- holds full parent chunks keyed by a
                          UUID generated at ingestion time. Lookups are O(1).
        - vectorstore   : ChromaDB       -- holds embedded child chunks; each
                          child Document carries 'doc_id' metadata pointing to
                          its parent UUID in the docstore.
        - parent_splitter: RecursiveCharacterTextSplitter with PARENT_CHUNK_SIZE;
                          splits the raw pages into parent-level segments.
        - child_splitter : RecursiveCharacterTextSplitter with CHILD_CHUNK_SIZE;
                          further splits each parent into child chunks for embedding.

    Ingestion step (add_documents):
        1. Apply parent_splitter to raw documents --> parent chunks.
        2. Assign a UUID to each parent chunk and store in docstore.
        3. Apply child_splitter to each parent chunk --> child chunks.
        4. Tag every child chunk with parent UUID in metadata['doc_id'].
        5. Embed all child chunks and persist to ChromaDB.

    Retrieval step (get_relevant_documents):
        1. Embed the query with the same embedding model.
        2. Find top-k most similar child chunks in ChromaDB (cosine similarity).
        3. Extract doc_id from each matched child's metadata.
        4. Deduplicate doc_ids (multiple children may share one parent).
        5. Fetch full parent Documents from InMemoryStore by doc_id.
        6. Return parent Documents to the chain for LLM generation.

    Args:
        documents  : Loaded and page-tagged PDF Documents.
        embeddings : Initialized HuggingFaceEmbeddings model.
        config     : PDRConfig with splitting and persistence settings.

    Returns:
        Populated ParentDocumentRetriever ready for retrieval queries.
    """
    # --- Parent splitter: large chunks that will be sent to the LLM --------
    parent_splitter = RecursiveCharacterTextSplitter(
        chunk_size=config.PARENT_CHUNK_SIZE,
        chunk_overlap=config.PARENT_CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", " ", ""],  # semantic priority order
        add_start_index=True,
    )

    # --- Child splitter: small chunks embedded for precise similarity -------
    child_splitter = RecursiveCharacterTextSplitter(
        chunk_size=config.CHILD_CHUNK_SIZE,
        chunk_overlap=config.CHILD_CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", " ", ""],
        add_start_index=True,
    )

    # --- Vector store (ChromaDB) for child chunk embeddings -----------------
    vectorstore = Chroma(
        collection_name=config.CHROMA_COLLECTION,
        embedding_function=embeddings,
        persist_directory=config.CHROMA_PERSIST_DIR,
    )

    # --- Document store for parent chunks -----------------------------------
    # InMemoryStore is a simple key-value store backed by a Python dict.
    # For production at scale, swap with RedisStore or SQLDocStore for
    # persistence and multi-process access.
    docstore = InMemoryStore()

    # --- Assemble the ParentDocumentRetriever --------------------------------
    retriever = ParentDocumentRetriever(
        vectorstore=vectorstore,
        docstore=docstore,
        child_splitter=child_splitter,
        parent_splitter=parent_splitter,
        search_kwargs={"k": config.RETRIEVER_K},
    )

    # --- Ingest documents ---------------------------------------------------
    # add_documents handles the full two-level split, UUID assignment,
    # embedding of children, and population of both stores atomically.
    logger.info(
        "Ingesting %d pages into ParentDocumentRetriever ...", len(documents)
    )
    logger.info(
        "  Parent chunk size : %d chars (overlap %d)",
        config.PARENT_CHUNK_SIZE, config.PARENT_CHUNK_OVERLAP,
    )
    logger.info(
        "  Child chunk size  : %d chars (overlap %d)",
        config.CHILD_CHUNK_SIZE, config.CHILD_CHUNK_OVERLAP,
    )

    retriever.add_documents(documents, ids=None)

    # Report how many child chunks were indexed in ChromaDB
    child_count = vectorstore._collection.count()
    logger.info(
        "Ingestion complete. Child chunks in ChromaDB: %d", child_count
    )

    return retriever

In [14]:

# ===========================================================================
# SECTION 6 -- AZURE OPENAI LLM INITIALIZATION
# ===========================================================================

def build_azure_llm(config: PDRConfig) -> AzureChatOpenAI:
    """
    Initialize the Azure OpenAI chat model.

    Required environment variables (set before running):
        AZURE_OPENAI_API_KEY     : Your Azure OpenAI resource key
        AZURE_OPENAI_ENDPOINT    : e.g. https://<resource>.openai.azure.com/
        AZURE_OPENAI_DEPLOYMENT  : Your deployment name (e.g. gpt-4o, gpt-4-turbo)
        AZURE_OPENAI_API_VERSION : API version string (e.g. 2024-02-15-preview)

    Azure OpenAI vs. OpenAI API:
        Azure OpenAI uses an endpoint-per-resource model. The deployment name
        is the model alias you configured in Azure AI Studio, not the model ID
        itself. The api_version parameter governs which feature set is exposed
        (streaming, function calling, structured outputs vary by version).

    Args:
        config: PDRConfig with Azure OpenAI credentials and model settings.

    Returns:
        Initialized AzureChatOpenAI instance.

    Raises:
        EnvironmentError: If required Azure credentials are not set.
    """
    missing = []
    if not config.AZURE_OPENAI_API_KEY:
        missing.append("AZURE_OPENAI_API_KEY")
    if not config.AZURE_OPENAI_ENDPOINT:
        missing.append("AZURE_OPENAI_ENDPOINT")
    if missing:
        raise EnvironmentError(
            f"Required Azure OpenAI environment variables are not set: {missing}\n"
            "Export them before running:\n"
            "  export AZURE_OPENAI_API_KEY='your-key'\n"
            "  export AZURE_OPENAI_ENDPOINT='https://your-resource.openai.azure.com/'\n"
            "  export AZURE_OPENAI_DEPLOYMENT='gpt-4o'\n"
            "  export AZURE_OPENAI_API_VERSION='2024-02-15-preview'"
        )

    logger.info(
        "Initializing Azure OpenAI LLM: deployment='%s', api_version='%s'",
        config.AZURE_OPENAI_DEPLOYMENT, config.AZURE_OPENAI_API_VERSION,
    )

    llm = AzureChatOpenAI(
        azure_deployment=config.AZURE_OPENAI_DEPLOYMENT,
        azure_endpoint=config.AZURE_OPENAI_ENDPOINT,
        api_key=config.AZURE_OPENAI_API_KEY,
        api_version=config.AZURE_OPENAI_API_VERSION,
        temperature=config.LLM_TEMPERATURE,
        max_tokens=config.LLM_MAX_TOKENS,
    )

    logger.info("Azure OpenAI LLM initialized successfully.")
    return llm

In [15]:

# ===========================================================================
# SECTION 7 -- RAG CHAIN CONSTRUCTION (LCEL)
# Combines the ParentDocumentRetriever with the Azure LLM into a single chain.
# ===========================================================================

def build_rag_chain(
    retriever: ParentDocumentRetriever,
    llm: AzureChatOpenAI,
    config: PDRConfig,
):
    """
    Assemble the PDR-based RAG chain using LangChain Expression Language (LCEL).

    The key difference from standard RAG is that the retriever here is the
    ParentDocumentRetriever, which internally returns PARENT-sized documents
    (1500 chars) to the LLM rather than CHILD-sized chunks (200 chars).

    This gives the LLM substantially more surrounding context per retrieved
    document. In contrast, standard RAG would pass the 200-char child chunks
    directly to the LLM, which often yields incomplete answers because the
    relevant sentence is cut off at the chunk boundary.

    Chain execution order per query:
        1. retriever.get_relevant_documents(question) -- async/parallel safe
        2. format_docs(docs) -- concatenate parent docs with source labels
        3. prompt.format(context=..., question=...) -- render prompt template
        4. llm(prompt) -- send to Azure OpenAI for completion
        5. StrOutputParser() -- extract string from AIMessage

    Args:
        retriever : Populated ParentDocumentRetriever.
        llm       : Initialized AzureChatOpenAI.
        config    : PDRConfig with RAG_PROMPT_TEMPLATE.

    Returns:
        Tuple of (rag_chain, retriever) -- chain for answering, retriever for
        source display.
    """
    prompt = ChatPromptTemplate.from_template(config.RAG_PROMPT_TEMPLATE)

    def format_docs(docs: List[Document]) -> str:
        """
        Format retrieved parent documents into a clean context block.

        Each document is labeled with its source paper name and page number
        so the LLM can cite provenance and the user can verify answers
        against the original research papers.
        """
        parts = []
        for i, doc in enumerate(docs, start=1):
            paper = doc.metadata.get("paper_name", doc.metadata.get("source", "Unknown"))
            page  = doc.metadata.get("page", "?")
            text  = doc.page_content.strip()
            parts.append(
                f"[Document {i} | Source: {paper} | Page: {page}]\n{text}"
            )
        return "\n\n" + ("=" * 60) + "\n\n".join(parts)

    # LCEL composition: context branch (retrieve + format) runs in parallel
    # with the question passthrough, then both are merged into the prompt.
    rag_chain = (
        {
            "context": retriever | format_docs,
            "question": RunnablePassthrough(),
        }
        | prompt
        | llm
        | StrOutputParser()
    )

    logger.info("PDR RAG chain assembled via LCEL.")
    return rag_chain, retriever

In [16]:


# ===========================================================================
# SECTION 8 -- QUERY INTERFACE
# ===========================================================================

def query_pdr(
    chain,
    retriever: ParentDocumentRetriever,
    question: str,
    show_sources: bool = True,
) -> str:
    """
    Execute a question against the PDR RAG pipeline.

    Displays the retrieved parent documents for transparency and
    returns the LLM-generated answer string.

    Args:
        chain        : LCEL RAG chain.
        retriever    : ParentDocumentRetriever for source display.
        question     : Natural language question.
        show_sources : If True, print source document metadata and excerpts.

    Returns:
        Answer string from the Azure OpenAI LLM.
    """
    logger.info("Query: '%s'", question)

    if show_sources:
        # Retrieve parent documents independently for display
        parent_docs = retriever.invoke(question)
        print("\n" + "=" * 70)
        print(f"QUERY: {question}")
        print("=" * 70)
        print(f"\nRETRIEVED PARENT DOCUMENTS ({len(parent_docs)}):")
        for i, doc in enumerate(parent_docs, start=1):
            paper = doc.metadata.get("paper_name", "Unknown")
            page  = doc.metadata.get("page", "?")
            snippet = doc.page_content[:300].replace("\n", " ").strip()
            doc_chars = len(doc.page_content)
            print(f"\n  [{i}] Source : {paper}  |  Page: {page}")
            print(f"       Length : {doc_chars} chars (full parent chunk sent to LLM)")
            print(f"       Excerpt: {snippet}...")

    # Execute the chain -- this triggers retrieval + LLM generation
    answer = chain.invoke(question)

    if show_sources:
        print(f"\nANSWER:\n{answer}")
        print("=" * 70 + "\n")

    return answer

In [17]:
 """
    End-to-end PDR RAG pipeline orchestrator.

    Steps:
        1. Download three arXiv PDFs (cached after first run).
        2. Load pages using PyPDFLoader.
        3. Initialize HuggingFace embedding model.
        4. Build ParentDocumentRetriever (ingest if new, or reload).
        5. Initialize Azure OpenAI LLM.
        6. Build LCEL RAG chain.
        7. Execute representative demo queries.
    """

'\n   End-to-end PDR RAG pipeline orchestrator.\n\n   Steps:\n       1. Download three arXiv PDFs (cached after first run).\n       2. Load pages using PyPDFLoader.\n       3. Initialize HuggingFace embedding model.\n       4. Build ParentDocumentRetriever (ingest if new, or reload).\n       5. Initialize Azure OpenAI LLM.\n       6. Build LCEL RAG chain.\n       7. Execute representative demo queries.\n   '

In [17]:
config = PDRConfig()

# Override with manual Azure OpenAI settings
config.AZURE_OPENAI_ENDPOINT = "https://dhanush-ai507.cognitiveservices.azure.com/"
config.AZURE_OPENAI_DEPLOYMENT = "gpt-4.1"
config.AZURE_OPENAI_API_VERSION = "2024-12-01-preview"
config.AZURE_OPENAI_API_KEY = "CGSB7TAAbigunwOWa1mKRqVtn6q1q6adAPwySS7B1TuxLNXKlhtoJQQJ99BKACYeBjFXJ3w3AAAAACOGeW9u"

In [18]:
pdf_paths = download_pdfs(config)

2026-03-26 22:08:08 | INFO     | parent_doc_retriever_rag | Cached PDF found: attention_is_all_you_need.pdf (2163.3 KB)
2026-03-26 22:08:08 | INFO     | parent_doc_retriever_rag | Cached PDF found: bert_pretraining_transformers.pdf (757.0 KB)
2026-03-26 22:08:08 | INFO     | parent_doc_retriever_rag | Cached PDF found: rag_knowledge_intensive_nlp.pdf (864.6 KB)


In [ ]:
# !pip install pypdf

In [29]:
import pypdf

In [19]:
documents = load_pdf_documents(pdf_paths)

2026-03-26 22:08:09 | INFO     | parent_doc_retriever_rag | Loading PDF: attention_is_all_you_need.pdf
2026-03-26 22:08:11 | INFO     | parent_doc_retriever_rag |   Loaded 15 pages from 'attention_is_all_you_need.pdf'
2026-03-26 22:08:11 | INFO     | parent_doc_retriever_rag | Loading PDF: bert_pretraining_transformers.pdf
2026-03-26 22:08:12 | INFO     | parent_doc_retriever_rag |   Loaded 16 pages from 'bert_pretraining_transformers.pdf'
2026-03-26 22:08:12 | INFO     | parent_doc_retriever_rag | Loading PDF: rag_knowledge_intensive_nlp.pdf
2026-03-26 22:08:12 | INFO     | parent_doc_retriever_rag |   Loaded 19 pages from 'rag_knowledge_intensive_nlp.pdf'
2026-03-26 22:08:12 | INFO     | parent_doc_retriever_rag | Total pages loaded across all PDFs: 50


In [34]:
# !pip install --upgrade transformers peft
# !pip install sentence-transformers

   ---------------------------------------- 0.0/10.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.7 MB ? eta -:--:--
    --------------------------------------- 0.3/10.7 MB ? eta -:--:--
    --------------------------------------- 0.3/10.7 MB ? eta -:--:--
   - -------------------------------------- 0.5/10.7 MB 525.5 kB/s eta 0:00:20
   - -------------------------------------- 0.5/10.7 MB 525.5 kB/s eta 0:00:20
   -- ------------------------------------- 0.8/10.7 MB 613.3 kB/s eta 0:00:17
   -- ------------------------------------- 0.8/10.7 MB 613.3 kB/s eta 0:00:17
   --- ------------------------------------ 1.0/10.7 MB 626.9 kB/s eta 0:00:16
   ---- ----------------------------------- 1.3/10.7 MB 677.8 kB/s eta 0:00:14
   ---- ----------------------------------- 1.3/10.7 MB 677.8 kB/s eta 0:00:14
   ----- ------------------

In [37]:
from sentence_transformers import SentenceTransformer


In [20]:
embeddings = build_embeddings(config)


2026-03-26 22:08:13 | INFO     | parent_doc_retriever_rag | Initializing embedding model: sentence-transformers/all-MiniLM-L6-v2 (device=cpu)
2026-03-26 22:08:13 | INFO     | sentence_transformers.SentenceTransformer | Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


C:\Users\dhanu\AppData\Local\Temp\ipykernel_37148\3781474231.py:29: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


2026-03-26 22:08:14 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"


2026-03-26 22:08:14 | WARNING  | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-03-26 22:08:14 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
2026-03-26 22:08:15 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-03-26 22:08:15 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-03-26 22:08:15 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/co

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6144.41it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2026-03-26 22:08:19 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-26 22:08:19 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config.json "HTTP/1.1 200 OK"
2026-03-26 22:08:20 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-26 22:08:21 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/tokenizer_config.json "HTTP/1.1 200 OK"
2026-03-26 22:08:21 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2/tree/main/additional_chat_templates?recursive=fals

In [27]:
retriever = build_parent_document_retriever(documents, embeddings, config)

2026-03-26 22:09:02 | INFO     | parent_doc_retriever_rag | Ingesting 50 pages into ParentDocumentRetriever ...
2026-03-26 22:09:02 | INFO     | parent_doc_retriever_rag |   Parent chunk size : 1500 chars (overlap 150)
2026-03-26 22:09:02 | INFO     | parent_doc_retriever_rag |   Child chunk size  : 200 chars (overlap 30)
2026-03-26 22:09:20 | INFO     | parent_doc_retriever_rag | Ingestion complete. Child chunks in ChromaDB: 2266


In [22]:
llm = build_azure_llm(config)

2026-03-26 22:08:25 | INFO     | parent_doc_retriever_rag | Initializing Azure OpenAI LLM: deployment='gpt-4.1', api_version='2024-12-01-preview'
2026-03-26 22:08:26 | INFO     | parent_doc_retriever_rag | Azure OpenAI LLM initialized successfully.


In [28]:
rag_chain, _ = build_rag_chain(retriever, llm, config)

2026-03-26 22:09:27 | INFO     | parent_doc_retriever_rag | PDR RAG chain assembled via LCEL.


In [24]:
demo_questions = [
        # Attention Is All You Need (1706.03762)
        "What is the motivation for replacing recurrent layers with self-attention in the Transformer architecture?",
        "How does multi-head attention work, and why is it preferred over single-head attention?",

        # BERT (1810.04805)
        "How does BERT differ from previous language models in terms of training direction?",
        "What are the two pre-training objectives used in BERT and how do they complement each other?",

        # RAG paper (2005.11401)
        "What problem does Retrieval-Augmented Generation solve compared to purely parametric language models?",
        "How does RAG combine a retrieval component with a generative model during inference?",
    ]

In [29]:
logger.info("Running %d demo queries ...", len(demo_questions))
for question in demo_questions:
    query_pdr(rag_chain, retriever, question, show_sources=True)


2026-03-26 22:09:27 | INFO     | parent_doc_retriever_rag | Running 6 demo queries ...
2026-03-26 22:09:27 | INFO     | parent_doc_retriever_rag | Query: 'What is the motivation for replacing recurrent layers with self-attention in the Transformer architecture?'

QUERY: What is the motivation for replacing recurrent layers with self-attention in the Transformer architecture?

RETRIEVED PARENT DOCUMENTS (2):

  [1] Source : Attention Is All You Need  |  Page: 9
       Length : 1453 chars (full parent chunk sent to LLM)
       Excerpt: 7 Conclusion In this work, we presented the Transformer, the first sequence transduction model based entirely on attention, replacing the recurrent layers most commonly used in encoder-decoder architectures with multi-headed self-attention. For translation tasks, the Transformer can be trained signi...

  [2] Source : Attention Is All You Need  |  Page: 5
       Length : 1439 chars (full parent chunk sent to LLM)
       Excerpt: corresponds to a sinusoid. 